# Interactive web maps with `InteractiveMap`

`digitalearth.interactive` is the **interactive-2D web-map tier**: the same pyramids data and auto-style
pipeline as the static `Map`, rendered through the **HoloViz stack** (GeoViews → HoloViews → Bokeh, plus
Datashader and Panel) so the result pans, zooms and hovers in the browser.

It is a *renderer, not a GIS engine*: every layer is built from pyramids-sourced arrays / GeoDataFrames and
all CRS work happens upstream in pyramids — never `xarray`/`rasterio`/`cartopy`. The engine import is **lazy**,
so `import digitalearth.interactive` works without the `interactive` extra; only a builder call needs it.

These notebooks use the bundled **Rhine-basin hydrology** data (gauges, river network, basin) and a
global **WorldClim temperature** stack — all real files under `examples/data/`.

> **▶ Run this notebook (Run All) to get interactive maps.** The `show(m)` helper displays an interactive
> **Bokeh** map — pan, zoom, hover, time-sliders. A saved `.ipynb` only shows interactive plots once the
> viewer runs their JavaScript: that means a **running kernel** (run the cells) or a **trusted** notebook. If a
> map looks blank when you just open the file, run the cell. A few layers with no Bokeh equivalent
> (projection, streamlines, wind barbs) render as static images via `show(m, static=True)`.

**Setup** — find the repo root, load the backends and define `show(...)`. Every notebook starts like this.

In [ ]:
import warnings
warnings.filterwarnings("ignore")            # hide backend-mismatch style warnings in the docs

from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import io
from IPython.display import Image
import matplotlib.pyplot as plt
import holoviews as hv
hv.extension("bokeh", "matplotlib")          # Bokeh = interactive; matplotlib = static (projection/barbs)

def show(m, static=False, dpi=96):
    """Display the map — **interactive Bokeh** by default (pan / zoom / hover).

    RUN THIS NOTEBOOK (Run All / Shift-Enter) to see the live interactive maps: a saved .ipynb only
    shows interactive plots once the viewer executes their JavaScript (a running kernel, or a trusted
    notebook). Pass ``static=True`` (used for the matplotlib-only layers — projection, streamlines,
    wind barbs, which have no Bokeh glyph) to render a static PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()                        # interactive Bokeh object

One call, one finished map: `InteractiveMap().field(...)`. `show(m)` renders it.

In [ ]:
from pyramids.dataset import Dataset
from digitalearth.interactive import InteractiveMap

dem = Dataset.read_file(str(DATA / "LisbonElevation.tif"))
m = InteractiveMap(crs=dem.epsg, title="Lisbon elevation")
m.field(dem, cmap="terrain")
show(m)

The builders **chain** (each returns the map) and compose as overlays — here a contour set over the shaded field.

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="elevation + contours")
m.field(dem, cmap="terrain").contours(dem, levels=8, color="black")
show(m)